# ⚡ Azure Sky Wind — Data Pipeline & Settlement Notebook

**Purpose:** One-stop notebook for updating SCED generation data, RTM market prices, and running VPPA settlement analysis for the Azure Sky Wind project (350 MW, ERCOT HB_NORTH).

**How to use:**
1. Mount your Google Drive (Section 1)
2. Run Setup (Section 2) to install dependencies
3. Pick the section you need — each one is independent after setup

| Parameter | Value |
|---|---|
| Resource ID | `AZURE_SKY_WIND_AGG` |
| Units | VORTEX_WIND1–4 |
| Capacity | 350 MW (nameplate) / 330.65 MW (modeled) |
| Location | 33.1534°N, 99.2847°W |
| Hub | HB_NORTH |
| Strike Price | $17.34/MWh |

---
## 1. Mount Google Drive & Set Paths

Upload your `sced_cache/` folder and price parquets to Drive, or point `BASE_DIR` at wherever your data lives.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# === CONFIGURE THESE PATHS ===
# Point this to your price_settlements repo on Drive
BASE_DIR = '/content/drive/MyDrive/price_settlements'

SCED_CACHE = f'{BASE_DIR}/sced_cache'
DATA_STATIC = f'{BASE_DIR}/data_static'

import os
os.makedirs(SCED_CACHE, exist_ok=True)
os.makedirs(DATA_STATIC, exist_ok=True)

print(f'Base directory: {BASE_DIR}')
print(f'SCED cache:    {SCED_CACHE}')
print(f'Files in cache: {len(os.listdir(SCED_CACHE)) if os.path.exists(SCED_CACHE) else 0}')

---
## 2. Install Dependencies & Imports

In [ ]:
!pip install -q gridstatus pandas pyarrow plotly openpyxl

import pandas as pd
import numpy as np
import gridstatus
import pyarrow.parquet as pq
from datetime import datetime, timedelta, date
import pytz
import warnings
import glob
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:,.2f}'.format)

ERCOT = gridstatus.Ercot()
CT = pytz.timezone('US/Central')
UTC = pytz.UTC

# Project constants
RESOURCE_ID = 'AZURE_SKY_WIND_AGG'
VORTEX_UNITS = ['VORTEX_WIND1', 'VORTEX_WIND2', 'VORTEX_WIND3', 'VORTEX_WIND4']
CAPACITY_MW = 350.0
MODELED_CAPACITY_MW = 330.65
HUB = 'HB_NORTH'
STRIKE_PRICE = 17.34  # $/MWh

print('Setup complete ✓')

---
## 3. SCED Data Pipeline

Fetch 60-day SCED disclosure data from ERCOT, aggregate VORTEX units, and save to cache.

> **Note:** ERCOT has a ~60-day disclosure lag. Data for the most recent 60 days won't be available.

### 3a. Check Data Status
See what dates we already have cached and identify gaps.

In [ ]:
def check_data_status():
    """Scan the SCED cache and report coverage."""
    daily_files = sorted(glob.glob(f'{SCED_CACHE}/*_{RESOURCE_ID}.parquet'))
    yearly_files = sorted(glob.glob(f'{SCED_CACHE}/{RESOURCE_ID}_*_full.parquet'))

    # Parse dates from daily filenames
    daily_dates = []
    for f in daily_files:
        basename = os.path.basename(f)
        try:
            d = datetime.strptime(basename[:10], '%Y-%m-%d').date()
            daily_dates.append(d)
        except ValueError:
            pass

    print(f'=== SCED Cache Status ===')
    print(f'Daily files:  {len(daily_dates)}')
    print(f'Yearly files: {len(yearly_files)}')

    if daily_dates:
        print(f'Date range:   {min(daily_dates)} → {max(daily_dates)}')

        # Find gaps
        all_dates = set(daily_dates)
        first, last = min(daily_dates), max(daily_dates)
        expected = set()
        d = first
        while d <= last:
            expected.add(d)
            d += timedelta(days=1)
        missing = sorted(expected - all_dates)
        print(f'Missing days: {len(missing)}')
        if missing and len(missing) <= 30:
            for m in missing:
                print(f'  - {m}')
        elif missing:
            print(f'  First 10: {missing[:10]}')
            print(f'  Last 10:  {missing[-10:]}')

    # Disclosure lag
    cutoff = (datetime.now() - timedelta(days=60)).date()
    print(f'\nDisclosure cutoff: {cutoff} (60-day lag)')
    if daily_dates:
        fetchable_gap = (cutoff - max(daily_dates)).days
        if fetchable_gap > 0:
            print(f'→ {fetchable_gap} days available to fetch')
        else:
            print(f'→ Cache is up to date with disclosure window')

    # Yearly file sizes
    for yf in yearly_files:
        df = pd.read_parquet(yf)
        yr = os.path.basename(yf).split('_')[4]
        print(f'\n  {yr}: {len(df):,} rows, {df["Time"].min()} → {df["Time"].max()}')
        if 'Base_Point_MW' in df.columns:
            bp_pct = df['Base_Point_MW'].notna().mean() * 100
            print(f'       Base_Point_MW coverage: {bp_pct:.1f}%')

    return daily_dates, missing if daily_dates else []

cached_dates, missing_dates = check_data_status()

### 3b. Time-Weighted Average (TWA) Aggregation
Core function that converts raw SCED telemetry into clean 15-minute intervals.

In [ ]:
def twa_aggregate(df_raw, date_val):
    """
    Time-weighted average of SCED telemetry into 15-min intervals.

    Args:
        df_raw: Raw SCED DataFrame with 'Interval Start', 'Telemetered Net Output', 'Base Point'
        date_val: The date being processed

    Returns:
        DataFrame with columns: Time, Actual_MW, MWh_interval, coverage, Base_Point_MW
    """
    if df_raw is None or len(df_raw) == 0:
        return None

    df = df_raw.copy()

    # Normalize column names
    col_map = {}
    for c in df.columns:
        cl = c.lower().strip()
        if 'telemetered' in cl:
            col_map[c] = 'telem_mw'
        elif 'base' in cl and 'point' in cl:
            col_map[c] = 'base_point'
        elif 'interval' in cl and 'start' in cl:
            col_map[c] = 'interval_start'
    df = df.rename(columns=col_map)

    if 'telem_mw' not in df.columns or 'interval_start' not in df.columns:
        return None

    df['interval_start'] = pd.to_datetime(df['interval_start'], utc=True)
    df = df.sort_values('interval_start').drop_duplicates(subset=['interval_start'])
    df['telem_mw'] = pd.to_numeric(df['telem_mw'], errors='coerce')

    has_bp = 'base_point' in df.columns
    if has_bp:
        df['base_point'] = pd.to_numeric(df['base_point'], errors='coerce')

    # Generate 15-min boundaries for the day
    day_start = pd.Timestamp(date_val, tz='UTC')
    day_end = day_start + pd.Timedelta(days=1)
    boundaries = pd.date_range(day_start, day_end, freq='15min')

    # Merge boundaries with actual timestamps
    all_ts = sorted(set(df['interval_start'].tolist() + boundaries.tolist()))
    df_idx = df.set_index('interval_start')
    df_merged = df_idx.reindex(all_ts).ffill()

    # Compute durations and energy
    df_merged['next_time'] = df_merged.index.to_series().shift(-1)
    df_merged['duration_sec'] = (df_merged['next_time'] - df_merged.index).dt.total_seconds()
    df_merged['duration_sec'] = df_merged['duration_sec'].clip(upper=3600)
    df_merged['energy_mwh'] = df_merged['telem_mw'] * df_merged['duration_sec'] / 3600
    if has_bp:
        df_merged['bp_energy'] = df_merged['base_point'] * df_merged['duration_sec'] / 3600

    # Group by 15-min floor
    df_merged['interval'] = df_merged.index.floor('15min')
    grouped = df_merged.groupby('interval')

    result = pd.DataFrame({
        'Time': grouped['energy_mwh'].apply(lambda x: x.name if hasattr(x, 'name') else x.index[0]),
        'total_energy': grouped['energy_mwh'].sum(),
        'total_duration': grouped['duration_sec'].sum(),
    })
    result.index = result.index.tz_localize(None).tz_localize('UTC') if result.index.tz is None else result.index
    result['Time'] = result.index
    result['total_hours'] = result['total_duration'] / 3600
    result['Actual_MW'] = np.where(result['total_hours'] > 0,
                                    result['total_energy'] / result['total_hours'], 0)
    result['MWh_interval'] = result['Actual_MW'] * 0.25
    result['coverage'] = result['total_duration'] / 900.0

    if has_bp:
        bp_energy = grouped['bp_energy'].sum()
        result['Base_Point_MW'] = np.where(result['total_hours'] > 0,
                                            bp_energy / result['total_hours'], np.nan)
    else:
        result['Base_Point_MW'] = np.nan

    # Filter to just this day's intervals
    result = result[(result['Time'] >= day_start) & (result['Time'] < day_end)]

    return result[['Time', 'Actual_MW', 'MWh_interval', 'coverage', 'Base_Point_MW']].reset_index(drop=True)

print('TWA function defined ✓')

### 3c. Fetch SCED Data for a Date Range
Downloads raw SCED data, aggregates VORTEX units, and saves daily parquets.

In [ ]:
def fetch_sced_day(target_date):
    """
    Fetch and aggregate all 4 VORTEX units for one day.
    Returns aggregated DataFrame or None on failure.
    """
    unit_dfs = []
    for unit in VORTEX_UNITS:
        try:
            raw = ERCOT.get_60_day_sced_disclosure(date=target_date)
            sced_df = raw.get('sced_gen_resource', raw.get('60_day_sced_disclosure', pd.DataFrame()))
            if isinstance(sced_df, dict):
                sced_df = list(sced_df.values())[0] if sced_df else pd.DataFrame()

            # Filter to this unit
            name_col = [c for c in sced_df.columns if 'resource' in c.lower() and 'name' in c.lower()]
            if name_col:
                unit_data = sced_df[sced_df[name_col[0]] == unit].copy()
            else:
                continue

            if len(unit_data) > 0:
                agg = twa_aggregate(unit_data, target_date)
                if agg is not None and len(agg) > 0:
                    unit_dfs.append(agg.set_index('Time'))
        except Exception as e:
            print(f'  Warning: {unit} failed for {target_date}: {e}')
            continue

    if not unit_dfs:
        return None

    # Sum across units
    combined = pd.concat(unit_dfs, axis=1, keys=range(len(unit_dfs)))
    result = pd.DataFrame(index=unit_dfs[0].index)
    result['Time'] = result.index

    # Sum Actual_MW and MWh_interval across units
    actual_cols = [(i, 'Actual_MW') for i in range(len(unit_dfs))]
    mwh_cols = [(i, 'MWh_interval') for i in range(len(unit_dfs))]
    cov_cols = [(i, 'coverage') for i in range(len(unit_dfs))]
    bp_cols = [(i, 'Base_Point_MW') for i in range(len(unit_dfs))]

    result['Actual_MW'] = combined[actual_cols].sum(axis=1)
    result['MWh_interval'] = combined[mwh_cols].sum(axis=1)
    result['coverage'] = combined[cov_cols].mean(axis=1)
    result['Base_Point_MW'] = combined[bp_cols].sum(axis=1, min_count=1)

    return result[['Time', 'Actual_MW', 'MWh_interval', 'coverage', 'Base_Point_MW']].reset_index(drop=True)


def fetch_date_range(start_date, end_date, skip_existing=True):
    """
    Fetch SCED data for a range of dates and save daily parquets.

    Args:
        start_date: First date (str 'YYYY-MM-DD' or date object)
        end_date: Last date (str 'YYYY-MM-DD' or date object)
        skip_existing: Skip dates that already have cached files
    """
    if isinstance(start_date, str):
        start_date = datetime.strptime(start_date, '%Y-%m-%d').date()
    if isinstance(end_date, str):
        end_date = datetime.strptime(end_date, '%Y-%m-%d').date()

    cutoff = (datetime.now() - timedelta(days=60)).date()
    if end_date > cutoff:
        print(f'⚠ Clamping end date to disclosure cutoff: {cutoff}')
        end_date = cutoff

    current = start_date
    success = 0
    skipped = 0
    failed = 0

    while current <= end_date:
        fname = f'{SCED_CACHE}/{current}_{RESOURCE_ID}.parquet'
        if skip_existing and os.path.exists(fname):
            skipped += 1
            current += timedelta(days=1)
            continue

        print(f'Fetching {current}...', end=' ')
        try:
            df = fetch_sced_day(current)
            if df is not None and len(df) > 0:
                df.to_parquet(fname, index=False)
                print(f'✓ {len(df)} rows, {df["Actual_MW"].mean():.1f} MW avg')
                success += 1
            else:
                print('✗ No data')
                failed += 1
        except Exception as e:
            print(f'✗ Error: {e}')
            failed += 1

        current += timedelta(days=1)

    print(f'\nDone: {success} fetched, {skipped} skipped, {failed} failed')

print('Fetch functions defined ✓')

### 3d. Run the Fetch
Edit the date range below and run. Set `skip_existing=False` to re-fetch dates you already have.

In [ ]:
# ✏️ EDIT THESE DATES
FETCH_START = '2026-03-01'
FETCH_END   = '2026-03-19'  # Will be clamped to 60-day disclosure cutoff

fetch_date_range(FETCH_START, FETCH_END, skip_existing=True)

### 3e. Rebuild Yearly Parquet
Consolidate daily cache files into a single yearly parquet for fast loading.

In [ ]:
def rebuild_yearly_parquet(year):
    """Consolidate daily parquets into one yearly file."""
    pattern = f'{SCED_CACHE}/{year}-*_{RESOURCE_ID}.parquet'
    files = sorted(glob.glob(pattern))
    print(f'Found {len(files)} daily files for {year}')

    if not files:
        print('No files to consolidate.')
        return None

    dfs = []
    for f in files:
        try:
            df = pd.read_parquet(f)
            dfs.append(df)
        except Exception as e:
            print(f'  Skipping {os.path.basename(f)}: {e}')

    combined = pd.concat(dfs, ignore_index=True)
    combined['Time'] = pd.to_datetime(combined['Time'], utc=True)
    combined = combined.sort_values('Time').drop_duplicates(subset=['Time'])

    out_path = f'{SCED_CACHE}/{RESOURCE_ID}_{year}_full.parquet'
    combined.to_parquet(out_path, index=False)

    print(f'Saved: {out_path}')
    print(f'  Rows: {len(combined):,}')
    print(f'  Range: {combined["Time"].min()} → {combined["Time"].max()}')
    print(f'  Avg MW: {combined["Actual_MW"].mean():.1f}')
    print(f'  Total MWh: {combined["MWh_interval"].sum():,.0f}')

    return combined

# ✏️ EDIT THE YEAR
yearly_df = rebuild_yearly_parquet(2026)

### 3f. Fill Specific Gaps
Target specific missing date ranges for re-fetch.

In [ ]:
# ✏️ ADD YOUR MISSING DATE RANGES HERE
MISSING_RANGES = [
    # ('2026-01-15', '2026-01-18'),
    # ('2026-02-03', '2026-02-05'),
]

for start, end in MISSING_RANGES:
    print(f'\n--- Filling gap: {start} → {end} ---')
    fetch_date_range(start, end, skip_existing=False)

---
## 4. RTM Price Data

Fetch and cache ERCOT Real-Time Market Settlement Point Prices (15-min intervals).

In [ ]:
def update_rtm_prices(year):
    """
    Fetch or update RTM SPP prices for a given year.
    Saves to BASE_DIR/ercot_rtm_{year}.parquet
    """
    out_path = f'{BASE_DIR}/ercot_rtm_{year}.parquet'

    # Check existing
    if os.path.exists(out_path):
        existing = pd.read_parquet(out_path)
        print(f'Existing: {len(existing):,} rows, {existing["Time"].max()}')
    else:
        existing = None
        print(f'No existing cache for {year}')

    print(f'Fetching RTM SPP for {year}...')
    try:
        rtm = ERCOT.get_rtm_spp(year=year)
    except Exception as e:
        print(f'Error fetching: {e}')
        return existing

    # Standardize columns
    if 'Interval Start' in rtm.columns:
        rtm = rtm.rename(columns={'Interval Start': 'Time'})
    if 'Location' not in rtm.columns:
        loc_col = [c for c in rtm.columns if 'location' in c.lower() or 'node' in c.lower()]
        if loc_col:
            rtm = rtm.rename(columns={loc_col[0]: 'Location'})
    if 'SPP' not in rtm.columns:
        spp_col = [c for c in rtm.columns if 'spp' in c.lower() or 'price' in c.lower()]
        if spp_col:
            rtm = rtm.rename(columns={spp_col[0]: 'SPP'})

    rtm['Time'] = pd.to_datetime(rtm['Time'], utc=True)

    # Add Central time
    rtm['Time_Central'] = rtm['Time'].dt.tz_convert('US/Central')

    rtm.to_parquet(out_path, index=False)
    print(f'Saved: {out_path}')
    print(f'  Rows: {len(rtm):,}')
    print(f'  Range: {rtm["Time"].min()} → {rtm["Time"].max()}')

    # Quick hub summary
    if 'Location' in rtm.columns and 'SPP' in rtm.columns:
        hub_avg = rtm.groupby('Location')['SPP'].mean()
        print(f'\n  Hub averages ($/MWh):')
        for hub, avg in hub_avg.items():
            print(f'    {hub}: ${avg:.2f}')

    return rtm

print('RTM price function defined ✓')

In [ ]:
# ✏️ EDIT THE YEAR(S) TO UPDATE
rtm_2025 = update_rtm_prices(2025)
rtm_2026 = update_rtm_prices(2026)

### 4a. Quick Price Chart

In [ ]:
def plot_rtm_prices(year, hub='HB_NORTH'):
    """Plot daily average RTM prices for a hub."""
    path = f'{BASE_DIR}/ercot_rtm_{year}.parquet'
    if not os.path.exists(path):
        print(f'No price data for {year}')
        return

    rtm = pd.read_parquet(path)
    rtm['Time'] = pd.to_datetime(rtm['Time'], utc=True)

    hub_data = rtm[rtm['Location'] == hub].copy()
    hub_data['date'] = hub_data['Time'].dt.date
    daily = hub_data.groupby('date').agg(
        avg_spp=('SPP', 'mean'),
        min_spp=('SPP', 'min'),
        max_spp=('SPP', 'max'),
        neg_intervals=('SPP', lambda x: (x < 0).sum())
    ).reset_index()

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=[f'{hub} Daily Avg RTM Price ({year})',
                                       'Negative Price Intervals per Day'],
                        row_heights=[0.7, 0.3])

    fig.add_trace(go.Scatter(x=daily['date'], y=daily['avg_spp'],
                            name='Avg $/MWh', line=dict(color='blue')), row=1, col=1)
    fig.add_hline(y=STRIKE_PRICE, line_dash='dash', line_color='red',
                  annotation_text=f'Strike: ${STRIKE_PRICE}', row=1, col=1)
    fig.add_trace(go.Bar(x=daily['date'], y=daily['neg_intervals'],
                        name='Neg intervals', marker_color='red'), row=2, col=1)

    fig.update_layout(height=600, showlegend=True)
    fig.show()

    # Stats
    print(f'\n{hub} {year} Summary:')
    print(f'  Avg price: ${daily["avg_spp"].mean():.2f}/MWh')
    print(f'  Days with negative prices: {(daily["neg_intervals"] > 0).sum()}')
    print(f'  Total negative intervals: {daily["neg_intervals"].sum():,}')

plot_rtm_prices(2026)

---
## 5. VPPA Settlement Analysis

Calculate net settlement amounts by merging SCED generation with RTM prices.

In [ ]:
def load_generation_data(year):
    """Load yearly aggregated SCED data."""
    path = f'{SCED_CACHE}/{RESOURCE_ID}_{year}_full.parquet'
    if not os.path.exists(path):
        print(f'No yearly file for {year}. Run Section 3e first.')
        return None
    df = pd.read_parquet(path)
    df['Time'] = pd.to_datetime(df['Time'], utc=True)
    return df


def load_rtm_hub(year, hub='HB_NORTH'):
    """Load RTM prices for a specific hub."""
    path = f'{BASE_DIR}/ercot_rtm_{year}.parquet'
    if not os.path.exists(path):
        print(f'No RTM data for {year}. Run Section 4 first.')
        return None
    rtm = pd.read_parquet(path)
    rtm['Time'] = pd.to_datetime(rtm['Time'], utc=True)
    hub_data = rtm[rtm['Location'] == hub][['Time', 'SPP']].copy()
    hub_data = hub_data.rename(columns={'SPP': 'Settlement_Point_Price'})
    return hub_data


def run_settlement(year, month=None, revenue_share=1.0, curtail_negative=True,
                   neg_price_floor=None, hub='HB_NORTH'):
    """
    Run VPPA settlement calculation.

    Args:
        year: Data year
        month: Optional month filter (1-12). None = full year.
        revenue_share: Upside revenue share (0-1). Downside is always 100%.
        curtail_negative: If True, zero out generation during negative prices.
        neg_price_floor: If set, floor SPP at this value (e.g., -3.00).
        hub: ERCOT hub for pricing.

    Returns:
        (merged_df, summary_dict)
    """
    gen = load_generation_data(year)
    prices = load_rtm_hub(year, hub)

    if gen is None or prices is None:
        return None, None

    # Merge on Time
    merged = pd.merge(gen, prices, on='Time', how='inner')
    merged = merged.sort_values('Time')

    if month:
        merged = merged[merged['Time'].dt.month == month]

    # Apply negative price floor
    if neg_price_floor is not None:
        merged['Effective_SPP'] = merged['Settlement_Point_Price'].clip(lower=neg_price_floor)
    else:
        merged['Effective_SPP'] = merged['Settlement_Point_Price']

    # Apply negative price curtailment
    if curtail_negative:
        merged['Effective_MWh'] = np.where(
            merged['Settlement_Point_Price'] < 0, 0, merged['MWh_interval'])
    else:
        merged['Effective_MWh'] = merged['MWh_interval']

    # Settlement calculation
    merged['Price_Diff'] = merged['Effective_SPP'] - STRIKE_PRICE

    # Revenue share: asymmetric — upside gets revenue_share, downside is 100%
    merged['Adj_Price_Diff'] = np.where(
        merged['Price_Diff'] > 0,
        merged['Price_Diff'] * revenue_share,
        merged['Price_Diff']  # buyer absorbs full downside
    )

    merged['Settlement_Amount'] = merged['Effective_MWh'] * merged['Adj_Price_Diff']

    # Summary
    period = f'{year}-{month:02d}' if month else str(year)
    hours = len(merged) * 0.25

    summary = {
        'period': period,
        'intervals': len(merged),
        'hours': hours,
        'total_generation_mwh': merged['MWh_interval'].sum(),
        'effective_generation_mwh': merged['Effective_MWh'].sum(),
        'capacity_factor': merged['Actual_MW'].mean() / CAPACITY_MW,
        'avg_rtm_price': merged['Settlement_Point_Price'].mean(),
        'gen_weighted_price': (
            (merged['Settlement_Point_Price'] * merged['MWh_interval']).sum()
            / merged['MWh_interval'].sum()
        ) if merged['MWh_interval'].sum() > 0 else 0,
        'total_received': merged.loc[merged['Settlement_Amount'] > 0, 'Settlement_Amount'].sum(),
        'total_paid': merged.loc[merged['Settlement_Amount'] < 0, 'Settlement_Amount'].sum(),
        'net_settlement': merged['Settlement_Amount'].sum(),
        'negative_price_intervals': (merged['Settlement_Point_Price'] < 0).sum(),
        'negative_price_pct': (merged['Settlement_Point_Price'] < 0).mean() * 100,
    }

    return merged, summary


def print_settlement(summary):
    """Pretty-print settlement summary."""
    if summary is None:
        return
    s = summary
    print(f'\n{"=" * 55}')
    print(f'  VPPA Settlement: {s["period"]}')
    print(f'{"=" * 55}')
    print(f'  Intervals:           {s["intervals"]:>10,}')
    print(f'  Generation:          {s["total_generation_mwh"]:>10,.0f} MWh')
    print(f'  Effective Gen:       {s["effective_generation_mwh"]:>10,.0f} MWh')
    print(f'  Capacity Factor:     {s["capacity_factor"]:>10.1%}')
    print(f'  Avg RTM Price:       {s["avg_rtm_price"]:>10.2f} $/MWh')
    print(f'  Gen-Wtd Price:       {s["gen_weighted_price"]:>10.2f} $/MWh')
    print(f'  Strike Price:        {STRIKE_PRICE:>10.2f} $/MWh')
    print(f'{"─" * 55}')
    print(f'  Amount Received:     {s["total_received"]:>10,.0f} $')
    print(f'  Amount Paid:         {s["total_paid"]:>10,.0f} $')
    print(f'  NET SETTLEMENT:      {s["net_settlement"]:>10,.0f} $')
    print(f'{"─" * 55}')
    print(f'  Neg Price Intervals: {s["negative_price_intervals"]:>10,} ({s["negative_price_pct"]:.1f}%)')
    print(f'{"=" * 55}')

print('Settlement functions defined ✓')

### 5a. Run Monthly Settlement

In [ ]:
# ✏️ EDIT YEAR AND MONTH
YEAR = 2026
MONTH = 1  # Set to None for full year

merged_df, summary = run_settlement(
    year=YEAR,
    month=MONTH,
    revenue_share=1.0,       # 100% revenue share
    curtail_negative=True,    # Zero gen during negative prices
    neg_price_floor=None,     # No floor (set to -3.0 to enable)
)

print_settlement(summary)

### 5b. Full Year Monthly Breakdown

In [ ]:
# ✏️ EDIT YEAR
YEAR = 2026

monthly_summaries = []
for m in range(1, 13):
    _, s = run_settlement(YEAR, month=m, curtail_negative=True)
    if s and s['intervals'] > 0:
        monthly_summaries.append(s)

if monthly_summaries:
    df_monthly = pd.DataFrame(monthly_summaries)
    print(df_monthly[['period', 'total_generation_mwh', 'capacity_factor',
                      'avg_rtm_price', 'net_settlement', 'negative_price_intervals']].to_string(index=False))

    # Chart
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=['Monthly Net Settlement ($)', 'Monthly Generation (MWh)'])

    colors = ['green' if v >= 0 else 'red' for v in df_monthly['net_settlement']]
    fig.add_trace(go.Bar(x=df_monthly['period'], y=df_monthly['net_settlement'],
                        marker_color=colors, name='Net Settlement'), row=1, col=1)
    fig.add_trace(go.Bar(x=df_monthly['period'], y=df_monthly['total_generation_mwh'],
                        marker_color='steelblue', name='Generation MWh'), row=2, col=1)

    fig.update_layout(height=600, showlegend=False)
    fig.show()

    # YTD totals
    print(f'\nYTD Net Settlement: ${df_monthly["net_settlement"].sum():,.0f}')
    print(f'YTD Generation:     {df_monthly["total_generation_mwh"].sum():,.0f} MWh')
    print(f'YTD Avg CF:         {df_monthly["capacity_factor"].mean():.1%}')

### 5c. Compare Against Invoice
Load settlement invoice actuals and compare with SCED-derived numbers.

In [ ]:
def compare_with_invoice(year, month):
    """Compare SCED-derived settlement with invoice actuals."""
    invoice_path = f'{DATA_STATIC}/Settlement_Invoice_Actuals.parquet'
    if not os.path.exists(invoice_path):
        print('No invoice data found. Upload Settlement_Invoice_Actuals.parquet to data_static/')
        return

    inv = pd.read_parquet(invoice_path)
    inv['Time'] = pd.to_datetime(inv['Time'])

    # Filter to month
    inv_month = inv[(inv['Time'].dt.year == year) & (inv['Time'].dt.month == month)]

    if len(inv_month) == 0:
        print(f'No invoice data for {year}-{month:02d}')
        return

    # SCED settlement
    _, sced_summary = run_settlement(year, month=month)

    # Invoice summary
    inv_gen = inv_month['Actual_MW'].sum() * 0.25  # MW to MWh
    inv_settlement = (inv_month['Actual_MW'] * 0.25 *
                      (inv_month['Settlement_Point_Price'] - STRIKE_PRICE)).sum()

    print(f'\n{"=" * 60}')
    print(f'  Invoice vs SCED Comparison: {year}-{month:02d}')
    print(f'{"=" * 60}')
    print(f'{"":<25} {"Invoice":>15} {"SCED":>15}')
    print(f'{"─" * 60}')
    print(f'{"Generation (MWh)":<25} {inv_gen:>15,.0f} {sced_summary["total_generation_mwh"]:>15,.0f}')
    print(f'{"Net Settlement ($)":<25} {inv_settlement:>15,.0f} {sced_summary["net_settlement"]:>15,.0f}')
    gen_diff = (sced_summary['total_generation_mwh'] - inv_gen) / inv_gen * 100 if inv_gen else 0
    stl_diff = (sced_summary['net_settlement'] - inv_settlement) / abs(inv_settlement) * 100 if inv_settlement else 0
    print(f'{"Gen Difference":<25} {"":>15} {gen_diff:>14.1f}%')
    print(f'{"Settlement Difference":<25} {"":>15} {stl_diff:>14.1f}%')
    print(f'{"=" * 60}')

# ✏️ EDIT YEAR AND MONTH
compare_with_invoice(2025, 12)

---
## 6. Performance Analysis

Check data quality, detect maintenance/outages, and compute key metrics.

In [ ]:
def performance_report(year, month=None):
    """Generate a quick performance report."""
    gen = load_generation_data(year)
    if gen is None:
        return

    if month:
        gen = gen[gen['Time'].dt.month == month]
        period = f'{year}-{month:02d}'
    else:
        period = str(year)

    hours = len(gen) * 0.25
    total_mwh = gen['MWh_interval'].sum()
    avg_mw = gen['Actual_MW'].mean()
    max_mw = gen['Actual_MW'].max()
    cf = avg_mw / CAPACITY_MW
    avg_coverage = gen['coverage'].mean()

    # Downtime detection
    zero_gen = gen[gen['Actual_MW'] < 2].copy()
    zero_hours = len(zero_gen) * 0.25

    # Daily capacity factors
    gen['date'] = gen['Time'].dt.date
    daily = gen.groupby('date').agg(
        daily_mwh=('MWh_interval', 'sum'),
        daily_avg_mw=('Actual_MW', 'mean'),
        daily_max_mw=('Actual_MW', 'max'),
        daily_coverage=('coverage', 'mean')
    ).reset_index()
    daily['daily_cf'] = daily['daily_avg_mw'] / CAPACITY_MW

    # Detect potential maintenance days (avg MW < 20% of capacity)
    maint_days = daily[daily['daily_cf'] < 0.05]

    print(f'\n{"=" * 50}')
    print(f'  Performance Report: {period}')
    print(f'{"=" * 50}')
    print(f'  Intervals:       {len(gen):>10,}')
    print(f'  Hours:           {hours:>10,.1f}')
    print(f'  Total MWh:       {total_mwh:>10,.0f}')
    print(f'  Avg MW:          {avg_mw:>10.1f}')
    print(f'  Max MW:          {max_mw:>10.1f}')
    print(f'  Capacity Factor: {cf:>10.1%}')
    print(f'  Data Coverage:   {avg_coverage:>10.1%}')
    print(f'  Zero-gen hours:  {zero_hours:>10,.1f} ({zero_hours/hours*100:.1f}%)')
    if len(maint_days) > 0:
        print(f'\n  Possible maintenance days ({len(maint_days)}):')
        for _, row in maint_days.iterrows():
            print(f'    {row["date"]}  CF={row["daily_cf"]:.1%}  Avg={row["daily_avg_mw"]:.1f} MW')
    print(f'{"=" * 50}')

    # Generation histogram
    fig = make_subplots(rows=2, cols=1,
                        subplot_titles=[f'Daily Generation ({period})',
                                       f'Output Distribution ({period})'])

    fig.add_trace(go.Bar(x=daily['date'], y=daily['daily_mwh'],
                        marker_color='steelblue', name='Daily MWh'), row=1, col=1)
    fig.add_trace(go.Histogram(x=gen['Actual_MW'], nbinsx=50,
                              marker_color='steelblue', name='MW Distribution'), row=2, col=1)

    fig.update_layout(height=600, showlegend=False)
    fig.show()

    return daily

# ✏️ EDIT YEAR AND OPTIONAL MONTH
daily_perf = performance_report(2026, month=None)

---
## 7. Export / Download

Export settlement results to Excel for sharing.

In [ ]:
def export_settlement_excel(year, output_name=None):
    """Export full year settlement to Excel with monthly tabs."""
    if output_name is None:
        output_name = f'{BASE_DIR}/Azure_Sky_Settlement_{year}.xlsx'

    with pd.ExcelWriter(output_name, engine='openpyxl') as writer:
        # Full year 15-min detail
        merged, _ = run_settlement(year)
        if merged is not None:
            export = merged[['Time', 'Actual_MW', 'MWh_interval', 'Settlement_Point_Price',
                            'Effective_MWh', 'Settlement_Amount']].copy()
            export['Time'] = export['Time'].dt.tz_convert('US/Central').dt.tz_localize(None)
            export.to_excel(writer, sheet_name='15min Detail', index=False)

        # Monthly summary
        summaries = []
        for m in range(1, 13):
            _, s = run_settlement(year, month=m)
            if s and s['intervals'] > 0:
                summaries.append(s)

        if summaries:
            df_sum = pd.DataFrame(summaries)
            df_sum.to_excel(writer, sheet_name='Monthly Summary', index=False)

    print(f'Saved: {output_name}')

    # Download in Colab
    try:
        from google.colab import files
        files.download(output_name)
    except:
        print('(Auto-download not available outside Colab)')

# ✏️ EDIT YEAR
export_settlement_excel(2026)

---
## 8. Quick Reference

| Task | Cell |
|---|---|
| Check what data we have | 3a |
| Fetch new SCED days | 3d |
| Rebuild yearly parquet | 3e |
| Fill specific gaps | 3f |
| Update RTM prices | 4 |
| Run monthly settlement | 5a |
| Full year breakdown | 5b |
| Compare vs invoice | 5c |
| Performance report | 6 |
| Export to Excel | 7 |

**Key gotchas:**
- SCED data has a **60-day disclosure lag** — you can't fetch the last 2 months
- All SCED/RTM data is **UTC** in parquets; converted to Central for display/export
- Invoice timestamps are **interval-ending** (Central time)
- Revenue share is **asymmetric**: buyer absorbs 100% of downside regardless of share %
- `MWh = MW × 0.25` for 15-min intervals